### Searching for the best models using the exhaustive search method

In [33]:
import numpy as np
from sklearn import linear_model, datasets
from sklearn.model_selection import GridSearchCV, cross_val_score

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data
target = iris.target

# Create a logistic regression model
logistic = linear_model.LogisticRegression(max_iter=500, solver='lbfgs')

# Create a range of possible values for the regularization strength hyperparameter
C = np.logspace(0, 4, 10)

# Create a dictionary of hyperparameters to search over
hyperparameters = dict(C=C)

# Create a grid search
grid_search = GridSearchCV(logistic, hyperparameters, cv=5, verbose=1, n_jobs=-1)

# Fit the grid search
best_model = grid_search.fit(features, target)

# Print the best model
print(best_model.best_estimator_)

print(cross_val_score(grid_search, features, target).mean())

Fitting 5 folds for each of 10 candidates, totalling 50 fits
LogisticRegression(C=np.float64(21.544346900318832), max_iter=500)
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
0.9666666666666668


### Random search for the best models

In [34]:
from scipy.stats import uniform
from sklearn import linear_model, datasets
from sklearn.model_selection import RandomizedSearchCV

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data
target = iris.target

# Create a logistic regression model
logistic = linear_model.LogisticRegression(max_iter=500, solver='lbfgs')

# Create a distribution of possible values for the regularization strength hyperparameter
C = uniform(loc=0.01, scale=4)

# Create a dictionary of hyperparameters to search over
hyperparameters = dict(C=C)

# Create a randomized search
random_search = RandomizedSearchCV(
    logistic,
    hyperparameters,
    random_state=1,
    n_iter=100,
    cv=5,
    verbose=0,
    n_jobs=-1,
)

# Perform the randomized search
best_model = random_search.fit(features, target)

# Print the best model and its optimal hyperparameters
print(best_model.best_estimator_)
print(best_model.best_params_)


LogisticRegression(C=np.float64(1.678088018810296), max_iter=500)
{'C': np.float64(1.678088018810296)}


### Finding the best model among multiple learning algorithms

In [35]:
import numpy as np
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Set a random seed for reproducibility
np.random.seed(0)

# Load the TRTS dataset
iris = datasets.load_iris()
features = iris.data
target = iris.target

# Create a pipeline with a random forest classifier
pipe = Pipeline([("classifier", RandomForestClassifier())])

# Define the search space for hyperparameter tuning
search_space = [
    {
        "classifier": [LogisticRegression(max_iter=500, solver='lbfgs')],
        "classifier__C": np.logspace(0, 4, 10),
    },
    {
        "classifier": [RandomForestClassifier()],
        "classifier__n_estimators": [10, 100, 1000],
        "classifier__max_features": [1, 2, 3],
    },
]

# Create a grid search with cross-validation
gridsearch = GridSearchCV(pipe, search_space, cv=5, verbose=1, n_jobs=-1)

# Fit the grid search to the data
best_model = gridsearch.fit(features, target)

# Print the best model and its optimal hyperparameters
print(best_model.best_estimator_)
print(best_model.best_params_)


Fitting 5 folds for each of 19 candidates, totalling 95 fits
Pipeline(steps=[('classifier',
                 LogisticRegression(C=np.float64(21.544346900318832),
                                    max_iter=500))])
{'classifier': LogisticRegression(max_iter=500), 'classifier__C': np.float64(21.544346900318832)}


### Finding the best model during the initial processing

In [36]:
import numpy as np
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set a random seed for reproducibility
np.random.seed(0)

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data
target = iris.target

# Create a preprocessing object that includes StandardScaler and PCA
preprocess = FeatureUnion([("std", StandardScaler()), ("pca", PCA())])

# Create a pipeline with preprocessing and logistic regression
pipe = Pipeline(
    [
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=1000, solver="lbfgs")),
    ]
)

# Define the search space for hyperparameter tuning
search_space = [
    {
        "preprocess__pca__n_components": [1, 2, 3],
        "classifier__C": np.logspace(0, 4, 10),
    }
]

# Create a grid search with cross-validation
clf = GridSearchCV(pipe, search_space, cv=5, verbose=0, n_jobs=-1)

# Fit the grid search to the data
best_model = clf.fit(features, target)

# Print the best model and its optimal hyperparameters
print(best_model.best_estimator_)
print(best_model.best_params_)

Pipeline(steps=[('preprocess',
                 FeatureUnion(transformer_list=[('std', StandardScaler()),
                                                ('pca', PCA(n_components=2))])),
                ('classifier',
                 LogisticRegression(C=np.float64(7.742636826811269),
                                    max_iter=1000))])
{'classifier__C': np.float64(7.742636826811269), 'preprocess__pca__n_components': 2}
